# 7장. 확률 보정

모델이 70%라고 예측한 일은 실제로 10번에 7번 일어나는가.

**준비:** 6장에서 쓴 경기도 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다.

## 7.1 보정 곡선: 점수를 확률로 읽어도 되는가

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

fit_on = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03"]], ignore_index=True)   # 식을 추정하는 자료
tune_on = snapshot("2020-04-03").reset_index(drop=True)                                                  # 보정에 쓰는 자료
test = snapshot("2023-04-03").reset_index(drop=True)                                                     # 시험 자료
y = test["폐업함"].to_numpy()
print(len(fit_on), len(tune_on), len(test), round(float(fit_on["폐업함"].mean()), 3), round(float(tune_on["폐업함"].mean()), 3), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return round(float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1))), 3)

coef = np.linalg.lstsq(columns(fit_on), fit_on["폐업함"].to_numpy(), rcond=None)[0]
line = columns(test) @ coef
print(auc(y, line), round(float(line.mean()), 3), round(float(line.min()), 3), round(float(line.max()), 3))

In [ ]:
def shelf(said, actual, edges):
    """예측 확률을 구간으로 나눠 구간마다 가게 수, 예측 확률의 평균, 실제로 폐업한 비율을 구한다."""
    t = pd.DataFrame({"구간": pd.cut(said, edges, include_lowest=True), "예측": said, "실제": actual})
    return t.groupby("구간", observed=True).agg(가게=("실제", "size"), 예측_평균=("예측", "mean"), 실제_비율=("실제", "mean")).round(3)

EDGES = [0, 0.12, 0.15, 0.18, 0.21, 0.25, 0.30, 1]
print(shelf(line, y, EDGES).to_string())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 7-2
a = shelf(line, y, EDGES)
se = np.sqrt(a["실제_비율"] * (1 - a["실제_비율"]) / a["가게"])
fig, ax = plt.subplots(figsize=(6.4, 5.2))
ax.plot([0, 0.45], [0, 0.45], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.errorbar(a["예측_평균"], a["실제_비율"], yerr=2 * se, color=NAVY, lw=1.4, capsize=3, zorder=2)
ax.scatter(a["예측_평균"], a["실제_비율"], s=a["가게"] / 9, color=to_rgba(NAVY, 0.3), edgecolor=NAVY, lw=1.0, zorder=3)
ax.text(0.44, 0.05, "원의 크기: 구간에 속한 가게 수(182곳 ~ 2,171곳)\n세로 막대: 실제 비율 ± 2 × 표준오차", fontsize=8, color=NAVY, ha="right", linespacing=1.5)
ax.text(0.36, 0.385, "예측 = 실제", fontsize=8, color="#6b7480", rotation=45, ha="center")
ax.text(0.02, 0.41, "대각선보다 위: 실제 폐업이 예측보다 많다", fontsize=8, color="#6b7480")
ax.text(0.44, 0.015, "대각선보다 아래: 실제 폐업이 예측보다 적다", fontsize=8, color="#6b7480", ha="right")
ax.set_xlim(0, 0.45); ax.set_ylim(0, 0.45); ax.set_aspect("equal")
ax.set_xlabel("예측 확률(구간의 평균)", fontsize=8)
ax.set_ylabel("실제로 폐업한 비율", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_7_2.png", dpi=300)
plt.show()

In [ ]:
on_fit = columns(fit_on) @ coef
young, old = (fit_on["업력"] < 3).to_numpy(), (fit_on["업력"] >= 10).to_numpy()
for name, mask in [("전체", np.ones(len(fit_on), dtype=bool)), ("업력 3년 미만", young), ("업력 10년 이상", old)]:
    print(name, round(float(on_fit[mask].mean() * 100), 2), round(float(fit_on.loc[mask, "폐업함"].mean() * 100), 2))

## 7.2 과신하는 예측: k-최근접 이웃과 브라이어 점수

In [ ]:
NEAR = ["업력", "주변", "주변폐업", "지번폐업", "위도", "경도"]

def stretch(df):
    z = df[NEAR].copy()
    z[["업력", "주변", "주변폐업"]] = np.log1p(z[["업력", "주변", "주변폐업"]])
    return z

seen = stretch(fit_on).dropna()
mu, sd = seen.mean(), seen.std()
A, closed = ((seen - mu) / sd).to_numpy(), fit_on.loc[seen.index, "폐업함"].to_numpy()

def neighbors(df, k=5):
    """여섯 특성에서 가장 가까운 k곳을 훈련 자료에서 찾아 그 가운데 폐업한 곳의 비율을 답한다."""
    B = ((stretch(df) - mu) / sd).fillna(0).to_numpy()
    out = np.empty(len(B))
    for i in range(0, len(B), 400):
        dist = ((B[i:i + 400, None, :] - A[None, :, :]) ** 2).sum(axis=2)
        out[i:i + 400] = closed[np.argpartition(dist, k, axis=1)[:, :k]].mean(axis=1)
    return out

five = neighbors(test)
print(auc(y, five), round(float(five.mean()), 3))
print(pd.DataFrame({"예측": five.round(1), "실제": y}).groupby("예측")["실제"].agg(["size", "mean"]).round(3).to_string())

In [ ]:
# 그림 7-3
b = pd.DataFrame({"예측": five.round(1), "실제": y}).groupby("예측")["실제"].agg(["size", "mean"])
fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.plot([0, 1], [0, 1], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.plot(b.index, b["mean"], color=BRICK, lw=1.4, zorder=2)
ax.scatter(b.index, b["mean"], s=np.maximum(b["size"] / 6, 12), color=to_rgba(BRICK, 0.35), edgecolor=BRICK, lw=1.0, zorder=3)
for x_, (n_, m_) in zip(b.index, b.to_numpy()):
    if abs(x_ - 0.6) < 1e-9:
        continue
    ax.text(x_, m_ + (0.075 if x_ <= 0.4 else 0.05), f"{int(n_):,}곳\n{m_ * 100:.1f}%", fontsize=7.5, color=BRICK, ha="center", linespacing=1.4)
ax.axhline(y.mean(), color=NAVY, lw=0.8, ls=(0, (2, 2)))
ax.text(0.3, y.mean() - 0.055, f"기준율 {y.mean() * 100:.1f}%", fontsize=8, color=NAVY, ha="center")
ax.text(0.52, 0.6, "예측 = 실제", fontsize=8, color="#6b7480", rotation=45, ha="center")
ax.annotate("폐업 확률 60%라고 예측한 354곳:\n실제로는 18.4%가 폐업했다", (0.6, b.loc[0.6, "mean"]), (0.46, 0.84), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.02, 1.0); ax.set_aspect("equal")
ax.set_xlabel("예측 확률(이웃 5곳 가운데 폐업한 비율)", fontsize=8)
ax.set_ylabel("실제로 폐업한 비율", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_7_3.png", dpi=300)
plt.show()

In [ ]:
def brier(actual, said):
    """예측 확률과 실제 결과(0 또는 1)의 차이를 제곱해 평균한다. 작을수록 좋다."""
    return round(float(((np.asarray(said) - actual) ** 2).mean()), 4)

base = np.full(len(y), fit_on["폐업함"].mean())
print(pd.DataFrame({"AUC": [0.5, auc(y, line), auc(y, five)],
                    "브라이어 점수": [brier(y, base), brier(y, line), brier(y, five)]},
                   index=["기준율만 답하기", "선형 회귀", "이웃 5곳"]).to_string())

In [ ]:
doubled = np.clip(line * 2, 0, 1)
print(auc(y, doubled), brier(y, doubled), round(float(doubled.mean()), 3))

In [ ]:
def parts(actual, said, bins=10):
    """브라이어 점수를 신뢰도, 분해능, 불확실성으로 나눈다. 예측 확률이 같은 가게끼리(값이 많으면 같은 수로 나눈 구간끼리) 묶는다."""
    said = np.asarray(said, dtype=float)
    group = pd.Series(said) if len(np.unique(said)) <= bins else pd.qcut(said, bins, duplicates="drop")
    t = pd.DataFrame({"묶음": np.asarray(group), "예측": said, "실제": actual}).groupby("묶음", observed=True).agg(n=("실제", "size"), p=("예측", "mean"), o=("실제", "mean"))
    w, rate = t["n"] / len(actual), actual.mean()
    rel, res, unc = (w * (t["p"] - t["o"]) ** 2).sum(), (w * (t["o"] - rate) ** 2).sum(), rate * (1 - rate)
    ece = (w * (t["p"] - t["o"]).abs()).sum()
    return [round(float(v), 4) for v in (rel, res, unc, rel - res + unc, ece)]

rows = {"기준율만 답하기": parts(y, base), "선형 회귀": parts(y, line), "이웃 5곳": parts(y, five), "선형 회귀 × 2": parts(y, doubled)}
print(pd.DataFrame(rows, index=["신뢰도", "분해능", "불확실성", "신뢰도 − 분해능 + 불확실성", "ECE"]).T.to_string())

## 7.3 재보정: 순위는 그대로 두고 확률만 고치기

In [ ]:
five_tune = neighbors(tune_on)
ruler = pd.DataFrame({"예측": five_tune.round(1), "실제": tune_on["폐업함"]}).groupby("예측")["실제"].agg(["size", "mean"]).round(3)
print(ruler.to_string())

In [ ]:
def reread(said):
    """이웃 5곳의 예측 확률을 2020년 자료에서 실제로 폐업한 비율로 바꾼다. 0.6 이상은 한 구간으로 합친다."""
    step = np.minimum(np.asarray(said).round(1), 0.6)
    known = np.minimum(five_tune.round(1), 0.6)
    rate = pd.Series(tune_on["폐업함"].to_numpy()).groupby(known).mean()
    return pd.Series(step).map(rate).to_numpy()

fixed = reread(five)
print(sorted(set(fixed.round(3).tolist())))
print(pd.DataFrame({"AUC": [auc(y, five), auc(y, fixed)], "브라이어 점수": [brier(y, five), brier(y, fixed)],
                    "ECE": [parts(y, five)[4], parts(y, fixed)[4]]}, index=["보정 전", "보정 후"]).to_string())

In [ ]:
# 그림 7-5
c = pd.DataFrame({"예측": fixed.round(3), "실제": y}).groupby("예측")["실제"].agg(["size", "mean"])
fig, ax = plt.subplots(figsize=(7, 4.0))
ax.plot([0, 1], [0, 1], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.plot(b.index, b["mean"], color=BRICK, lw=1.2, marker="s", ms=5, label="보정 전: 0%부터 100%까지 여섯 가지 값")
ax.plot(c.index, c["mean"], color=TEAL, lw=1.6, marker="o", ms=6, label="보정 후: 16.0%부터 22.3%까지 네 가지 값")
for (x0, y0), x1 in zip(zip([0.0, 0.2, 0.4, 0.6], b["mean"].iloc[:4]), sorted(c.index)):
    ax.annotate("", (x1 + 0.006, c.loc[x1, "mean"]), (x0, y0), arrowprops={"arrowstyle": "->", "color": GRAY, "lw": 0.7, "ls": (0, (2, 2))})
ax.text(0.62, 0.52, "예측 = 실제", fontsize=8, color="#6b7480", rotation=22)
ax.set_xlim(-0.03, 1.03); ax.set_ylim(0, 0.6)
ax.set_xlabel("예측 확률", fontsize=8)
ax.set_ylabel("실제로 폐업한 비율(2023년 시험 자료)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_7_5.png", dpi=300)
plt.show()

In [ ]:
def isotonic(x, t):
    """x가 커질 때 줄어들지 않는 계단 함수 가운데 t와의 제곱 오차가 가장 작은 것을 구한다(PAV 알고리즘)."""
    order = np.argsort(x, kind="stable")
    stack = []                                   # [합, 개수]인 묶음들
    for v in np.asarray(t, dtype=float)[order]:
        stack.append([v, 1])
        while len(stack) > 1 and stack[-2][0] / stack[-2][1] >= stack[-1][0] / stack[-1][1]:
            s, c = stack.pop()
            stack[-1][0] += s
            stack[-1][1] += c
    return np.asarray(x)[order], np.concatenate([np.full(c, s / c) for s, c in stack])

line_tune = columns(tune_on) @ coef
xs, steps = isotonic(line_tune, tune_on["폐업함"].to_numpy())
line_iso = np.interp(line, xs, steps)
print(len(np.unique(steps)), np.unique(steps).round(3).tolist())
print(pd.DataFrame({"AUC": [auc(y, line), auc(y, line_iso)], "브라이어 점수": [brier(y, line), brier(y, line_iso)],
                    "ECE": [parts(y, line)[4], parts(y, line_iso)[4]]}, index=["선형 회귀", "등위 회귀로 보정"]).to_string())

In [ ]:
# 그림 7-6
fig, ax = plt.subplots(figsize=(7, 4.0))
ax.plot([0, 0.6], [0, 0.6], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.step(xs, steps, where="post", color=TEAL, lw=2.0, label="등위 회귀: 2020년 자료에서 구한 계단 함수")
ax.hist(line_tune, bins=np.arange(0.08, 0.6, 0.01), weights=np.full(len(line_tune), 0.6 / 1400), color=to_rgba(NAVY, 0.15), edgecolor="none", label="2020년 자료의 점수 분포(높이는 상대적인 수)")
ax.text(0.5, 0.465, "보정 전 = 보정 후", fontsize=8, color="#6b7480", rotation=33)
ax.set_xlim(0.08, 0.6); ax.set_ylim(0, 0.6)
ax.set_xlabel("선형 회귀의 점수(보정 전)", fontsize=8)
ax.set_ylabel("보정한 확률", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_7_6.png", dpi=300)
plt.show()

## 7.4 집단별 보정과 자료에 들어 있는 미래 정보

In [ ]:
def side_by_side(name, mask):
    return (name, int(mask.sum()), round(float(line[mask].mean() * 100), 1), round(float(y[mask].mean() * 100), 1))

title = test["사업장명"]
rows = [side_by_side("전체", np.ones(len(y), dtype=bool)),
        side_by_side("업력 3년 미만", (test["업력"] < 3).to_numpy()),
        side_by_side("업력 3~10년", ((test["업력"] >= 3) & (test["업력"] < 10)).to_numpy()),
        side_by_side("업력 10년 이상", (test["업력"] >= 10).to_numpy()),
        side_by_side("상호에 마라", title.str.contains("마라").to_numpy()),
        side_by_side("상호에 양꼬치", title.str.contains("양꼬치").to_numpy()),
        side_by_side("상호에 반점", title.str.contains("반점").to_numpy())]
by_group = pd.DataFrame(rows, columns=["집단", "가게", "예측(%)", "실제(%)"])
print(by_group.to_string(index=False))

In [ ]:
# 그림 7-7
fig, ax = plt.subplots(figsize=(7, 4.0))
ypos = np.arange(len(by_group))[::-1]
for yi, (_, row) in zip(ypos, by_group.iterrows()):
    lo, hi = sorted([row["예측(%)"], row["실제(%)"]])
    ax.plot([lo, hi], [yi, yi], color=GRAY, lw=2.0, zorder=1)
    ax.scatter([row["예측(%)"]], [yi], s=46, color=NAVY, zorder=3)
    ax.scatter([row["실제(%)"]], [yi], s=46, color=BRICK, zorder=3)
    gap = row["실제(%)"] - row["예측(%)"]
    ax.text(31.5, yi, f"예측 {row['예측(%)']:.1f}%  실제 {row['실제(%)']:.1f}%  (차이 {gap:+.1f}%p)", va="center", ha="left", fontsize=8, color="#44505c")
ax.scatter([], [], s=46, color=NAVY, label="예측 확률의 평균")
ax.scatter([], [], s=46, color=BRICK, label="실제로 폐업한 비율")
ax.set_yticks(ypos, [f"{g} ({n:,}곳)" for g, n in zip(by_group["집단"], by_group["가게"])], fontsize=8)
ax.set_xlim(8, 52); ax.set_ylim(-0.7, len(by_group) + 0.3)
ax.set_xticks([10, 15, 20, 25, 30])
ax.set_xlabel("%", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right", ncol=2, bbox_to_anchor=(1.0, 1.04))
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_7_7.png", dpi=300)
plt.show()

In [ ]:
new = (test["구분"] == "공백 없음").to_numpy()
print(pd.DataFrame([side_by_side("시군명에 공백 없음", new), side_by_side("시군명에 공백 있음", ~new)],
                   columns=["집단", "가게", "예측(%)", "실제(%)"]).to_string(index=False))
print(auc(y, new.astype(float)), auc(y, line + new))

In [ ]:
rows = []
for when, d in [("2011~2017년 기준일", fit_on), ("2023년 기준일", test)]:
    has = d["사업장명"].str.contains("마라")
    rows.append((when, int(has.sum()), round(float(d.loc[has, "폐업함"].mean() * 100), 1), round(float(d.loc[~has, "폐업함"].mean() * 100), 1)))
mara = pd.DataFrame(rows, columns=["자료", "상호에 마라가 있는 줄", "그 폐업률(%)", "나머지의 폐업률(%)"])
print(mara.to_string(index=False))

In [ ]:
# 그림 7-8
labels = ["시군명에 공백 없음\n(2023년 시험 자료 4,107곳)", "시군명에 공백 있음\n(2023년 시험 자료 2,204곳)", "상호에 ‘마라’\n(2011~17년 기준일 371줄)", "상호에 ‘마라’\n(2023년 기준일 982곳)"]
actual = [float(y[new].mean() * 100), float(y[~new].mean() * 100), mara.loc[0, "그 폐업률(%)"], mara.loc[1, "그 폐업률(%)"]]
ref = [float(y.mean() * 100)] * 2 + [mara.loc[0, "나머지의 폐업률(%)"], mara.loc[1, "나머지의 폐업률(%)"]]
x = np.arange(4)
fig, ax = plt.subplots(figsize=(7, 4.1))
ax.bar(x - 0.2, ref, width=0.4, color=to_rgba(GRAY, 0.45), edgecolor=GRAY, lw=0.9, label="비교 기준(왼쪽 둘: 시험 자료 전체, 오른쪽 둘: 상호에 ‘마라’가 없는 가게)")
ax.bar(x + 0.2, actual, width=0.4, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.9, label="그 집단에서 실제로 폐업한 비율")
for xi, (r, a_) in enumerate(zip(ref, actual)):
    ax.text(xi - 0.2, r + 0.7, f"{r:.1f}%", ha="center", fontsize=8, color="#6b7480")
    ax.text(xi + 0.2, a_ + 0.7, f"{a_:.1f}%", ha="center", fontsize=8, color=BRICK)
ax.axvline(1.5, color=GRAY, lw=0.6, ls=(0, (2, 2)))
ax.set_xticks(x, labels, fontsize=7.8)
ax.set_ylim(0, 38)
ax.set_ylabel("3년 안에 폐업한 비율(%)", fontsize=8)
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=7.8, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_7_8.png", dpi=300)
plt.show()